In [1]:
print("PDF RAG GEMINI")

PDF RAG GEMINI


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [4]:
llm=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3
    )

In [5]:
llm.invoke("What is Science").content

'Science is a **systematic enterprise that builds and organizes knowledge in the form of testable explanations and predictions about the universe.**\n\nIt\'s both:\n\n1.  **A Body of Knowledge:** The accumulated facts, theories, laws, and models that describe how the natural world works.\n2.  **A Process (The Scientific Method):** The rigorous, evidence-based approach used to acquire and refine that knowledge.\n\nHere are the key characteristics and principles of science:\n\n*   **Empirical:** Science relies on observation and experimentation. Knowledge is derived from sensory experience and measurable data, not just pure reason or belief.\n*   **Testable/Falsifiable:** Scientific hypotheses and theories must be capable of being proven wrong through observation or experiment. If an idea cannot, in principle, be tested or disproven, it falls outside the realm of science.\n*   **Objective (aspiration):** While individual scientists may have biases, the scientific method aims to minimize 

In [6]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


C:\Users\HP\AppData\Local\Temp\ipykernel_1556\953043586.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader=PyMuPDFLoader("Hamza_Personal_Profile.pdf")
document=loader.load()

In [9]:
document[0].page_content

"Muhammad Hamza - Personal Profile\nHi, I'm Hamza\nData Scientist | Machine Learning Enthusiast | Developer\n- Education:\n  - BS Data Science - Emerson University, Multan\n  - GPA (1st Semester): 4.00/4.00\n  - Currently in 2nd year; will be in 3rd semester in September 2025.\n- Languages: English, Urdu, Punjabi\n- Passionate about Machine Learning, Deep Learning (CNNs), and Data Science\n- Goal: Become a great coder and innovator in tech.\n- Currently working on Arduino IoT projects, ML, DL\n------------------------------------------------------------\nMy Projects\nMachine Learning & AI Projects\n- Titanic Survival  Predict who survived using ML\n- Score Prediction  Estimate academic scores based on input features\n- Banana Quality Classification  Detect banana quality using image classification (CNN)\n- Match Maker  Predict compatibility between individuals\n- Weather Prediction  Predict temperature, humidity using regression\n- Student Attendance System (ID Card based)  Track stude

In [10]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=30)

In [12]:
chunks=text_splitter.split_documents(document)

In [14]:
for c in chunks:
    print(c)

page_content='Muhammad Hamza - Personal Profile
Hi, I'm Hamza
Data Scientist | Machine Learning Enthusiast | Developer
- Education:
  - BS Data Science - Emerson University, Multan
  - GPA (1st Semester): 4.00/4.00
  - Currently in 2nd year; will be in 3rd semester in September 2025.' metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20250812091113', 'source': 'Hamza_Personal_Profile.pdf', 'file_path': 'Hamza_Personal_Profile.pdf', 'total_pages': 3, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20250812091113', 'page': 0}
page_content='- Languages: English, Urdu, Punjabi
- Passionate about Machine Learning, Deep Learning (CNNs), and Data Science
- Goal: Become a great coder and innovator in tech.
- Currently working on Arduino IoT projects, ML, DL
------------------------------------------------------------
My Projects' metadata={'producer

In [16]:
vectorstores=FAISS.from_documents(chunks,embed)

In [18]:
retriever=vectorstores.as_retriever(search_type='similarity',search_kwargs={'k':3})

In [31]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
memory=ConversationBufferMemory()

C:\Users\HP\AppData\Local\Temp\ipykernel_1556\2314992378.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory()


In [23]:
template="""You are PDF Bot. you will get context always reply from given context. Reply Should not be very large
or very small. Just 50 words untill or unless you have to explain something in details.
Here is user input: {question}
Here is Context: {context}"""
prompt=PromptTemplate(template=template,input_variables=['question','context'])

In [37]:
pdf_bot=RetrievalQA.from_chain_type(llm=llm,retriever=retriever,chain_type_kwargs={'prompt':prompt},memory=memory,chain_type="stuff")

In [38]:
response=pdf_bot.invoke("Who is Hamza?")

In [39]:
print('------PDF BOT-------')
print(response['result'])

------PDF BOT-------
Muhammad Hamza is a Data Scientist, Machine Learning Enthusiast, and Developer. He is a 2nd-year BS Data Science student at Emerson University, Multan, holding a 4.00 GPA. He is passionate about Machine Learning and Deep Learning, currently working on Arduino IoT, ML, and DL projects. His goal is to become a great coder and innovator.
